# 차량 차종 분류 학습/검증 노트북

현재 기준 모델은 `yolo26s_cls_full-27-2`입니다.

이 노트북에서 하는 일:
- Drive 마운트
- `/content/car-cls` 데이터셋 점검
- `full-27-2` 학습 재개
- `full-27-2`만 실제 현장 crop 평가
- 사용자가 올린 CCTV 차량 이미지 4장 전용 추론

주의: `last.pt`는 기본적으로 에포크 단위로 저장됩니다. `save_period=1`을 켜도 한 에포크 중간 저장은 되지 않습니다.


In [ ]:
!pip install -q ultralytics opencv-python pillow pyyaml tqdm pandas matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 4.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
from pathlib import Path
import random
import re
import shutil

import pandas as pd
from ultralytics import YOLO
from IPython.display import display, Image

drive.mount('/content/drive', force_remount=True)

PROJECT_DIR = Path('/content/car-cls')
PROJECT_ZIP = Path('/content/drive/MyDrive/car-cls.zip')
PROJECT_TAR = Path('/content/drive/MyDrive/car-cls.tar')
RUNS_DRIVE = Path('/content/drive/MyDrive/runs/vehicle_cls')

DATASET_DIR = PROJECT_DIR / 'data' / 'vehicle_cls_crops_sampled'

BEST_RUN_NAME = 'yolo26s_cls_full-27-2'
BEST_RUN_DIR = RUNS_DRIVE / BEST_RUN_NAME
BEST_WEIGHTS = BEST_RUN_DIR / 'weights' / 'best.pt'
LAST_WEIGHTS = BEST_RUN_DIR / 'weights' / 'last.pt'

EVAL_OUT = Path('/content/drive/MyDrive/runs/real_field_crop_eval_1000_full27_2')
EVAL_OUT.mkdir(parents=True, exist_ok=True)

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

print('PROJECT_DIR:', PROJECT_DIR)
print('DATASET_DIR:', DATASET_DIR)
print('RUNS_DRIVE:', RUNS_DRIVE)
print('BEST_WEIGHTS:', BEST_WEIGHTS)
print('LAST_WEIGHTS:', LAST_WEIGHTS)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


ValueError: mount failed

## 1. 데이터셋 준비/점검

학습 전에 `train` 클래스 수가 정상인지 반드시 확인합니다.
`train classes`가 20여 개처럼 작게 나오면 `/content/car-cls` 압축 해제가 덜 된 상태입니다.


In [ ]:
FORCE_REEXTRACT_PROJECT = False

if FORCE_REEXTRACT_PROJECT and PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

if not DATASET_DIR.exists():
    if PROJECT_ZIP.exists():
        print('car-cls.zip 압축 해제 중...')
        !unzip -q -o "$PROJECT_ZIP" -d /content
    elif PROJECT_TAR.exists():
        print('car-cls.tar 압축 해제 중...')
        !tar -xf "$PROJECT_TAR" -C /content
    else:
        raise FileNotFoundError('Drive에서 car-cls.zip 또는 car-cls.tar를 찾지 못했습니다.')

def count_split(split_name):
    split_dir = DATASET_DIR / split_name
    if not split_dir.exists():
        return 0, 0
    class_dirs = [p for p in split_dir.iterdir() if p.is_dir()]
    image_count = sum(1 for p in split_dir.rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXTS)
    return len(class_dirs), image_count

for split in ['train', 'val', 'test']:
    n_classes, n_images = count_split(split)
    print(f'{split} classes={n_classes} images={n_images}')

train_classes, train_images = count_split('train')
if train_classes < 300:
    raise RuntimeError(
        'train 클래스 수가 너무 적습니다. FORCE_REEXTRACT_PROJECT=True로 바꾸고 이 셀부터 다시 실행하세요.'
    )

car-cls.tar 압축 해제 중...
train classes=410 images=1232208
val classes=410 images=230071
test classes=410 images=75353


## 2. full-27-2 학습 재개

아래 셀은 `full-27-2`의 `last.pt`를 찾아 이어서 학습합니다.
학습 전 `best.pt`, `last.pt`를 Drive에 백업합니다.


In [ ]:
TARGET_EPOCHS = 50
PATIENCE = 10
BATCH = 16
WORKERS = 2
IMG_SIZE = 224

def backup_checkpoint(path):
    if not path.exists():
        return None
    backup_dir = RUNS_DRIVE / '_checkpoint_backups'
    backup_dir.mkdir(parents=True, exist_ok=True)
    dst = backup_dir / f'{BEST_RUN_NAME}_{path.name}_before_resume.pt'
    shutil.copy2(path, dst)
    return dst

if not LAST_WEIGHTS.exists():
    candidates = sorted(RUNS_DRIVE.glob('yolo26s_cls_full-27-2*/weights/last.pt'), key=lambda p: p.stat().st_mtime, reverse=True)
    if candidates:
        LAST_WEIGHTS = candidates[0]
        BEST_RUN_DIR = LAST_WEIGHTS.parents[1]
        BEST_WEIGHTS = BEST_RUN_DIR / 'weights' / 'best.pt'
        print('대체 last.pt를 찾았습니다:', LAST_WEIGHTS)
    else:
        raise FileNotFoundError('full-27-2 계열 last.pt를 찾지 못했습니다.')

print('학습 재개 기준 last.pt:', LAST_WEIGHTS)
print('평가 우선 best.pt:', BEST_WEIGHTS if BEST_WEIGHTS.exists() else '없음')

for ckpt in [BEST_WEIGHTS, LAST_WEIGHTS]:
    backup = backup_checkpoint(ckpt)
    if backup:
        print('백업 완료:', backup)

In [ ]:
model = YOLO(str(LAST_WEIGHTS), task='classify')

train_results = model.train(
    resume=True,
    data=str(DATASET_DIR),
    epochs=TARGET_EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=0,
    workers=WORKERS,
    patience=PATIENCE,
    cache=False,
    save=True,
    save_period=1,
)

print('학습 재개 완료')
print(train_results)

## 3. full-27-2만 실제 현장 crop 평가

이 평가는 detector를 거치지 않고, 이미 차량 bbox만큼 crop된 실제 현장 이미지로 분류모델 자체 성능을 봅니다.


In [ ]:
SOURCE_SPLIT_DIR = DATASET_DIR / 'test' if (DATASET_DIR / 'test').exists() else DATASET_DIR / 'val'
MAX_TOTAL = 1000
SEED = 42
SAMPLE_DIR = EVAL_OUT / 'sample_1000_real_field_crops'

if SAMPLE_DIR.exists():
    shutil.rmtree(SAMPLE_DIR)
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
class_dirs = sorted([p for p in SOURCE_SPLIT_DIR.iterdir() if p.is_dir()])
items_by_class = []
for cls_dir in class_dirs:
    imgs = [p for p in cls_dir.rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXTS]
    random.shuffle(imgs)
    items_by_class.append((cls_dir.name, imgs))

selected = []
round_idx = 0
while len(selected) < MAX_TOTAL:
    added = False
    for cls_name, imgs in items_by_class:
        if round_idx < len(imgs):
            selected.append((cls_name, imgs[round_idx]))
            added = True
            if len(selected) >= MAX_TOTAL:
                break
    if not added:
        break
    round_idx += 1

for cls_name, src in selected:
    dst_dir = SAMPLE_DIR / cls_name
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / src.name
    if dst.exists():
        dst = dst_dir / f'{src.stem}_{abs(hash(str(src))) % 100000}{src.suffix.lower()}'
    shutil.copy2(src, dst)

sample_images = sorted([p for p in SAMPLE_DIR.rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXTS])
print('샘플 폴더:', SAMPLE_DIR)
print('샘플 이미지 수:', len(sample_images))
print('샘플 클래스 수:', len([p for p in SAMPLE_DIR.iterdir() if p.is_dir()]))

FileNotFoundError: [Errno 2] No such file or directory: '/content/car-cls/data/vehicle_cls_crops_sampled/val'

In [ ]:
def norm_name(value):
    value = Path(str(value)).stem.lower()
    value = re.sub(r'_\d+$', '', value)
    value = re.sub(r'[^a-z0-9]+', '_', value)
    value = re.sub(r'_+', '_', value).strip('_')
    return value

def is_close_match(pred, expected):
    pred = norm_name(pred)
    expected = norm_name(expected)
    if pred == expected:
        return True
    if pred.startswith(expected + '_') or expected.startswith(pred + '_'):
        return True
    pred_tokens = pred.split('_')
    exp_tokens = expected.split('_')
    if len(pred_tokens) >= 2 and len(exp_tokens) >= 2:
        return pred_tokens[:2] == exp_tokens[:2]
    return False

CLS_WEIGHTS = BEST_WEIGHTS if BEST_WEIGHTS.exists() else LAST_WEIGHTS
print('평가 모델:', CLS_WEIGHTS)

model = YOLO(str(CLS_WEIGHTS), task='classify')
rows = []
for img_path in sample_images:
    expected = img_path.parent.name
    result = model.predict(source=str(img_path), imgsz=IMG_SIZE, verbose=False)[0]
    top1 = int(result.probs.top1)
    pred = result.names[top1]
    conf = float(result.probs.top1conf)
    if norm_name(pred) == norm_name(expected):
        label = 'true'
    elif is_close_match(pred, expected):
        label = 'close'
    else:
        label = 'false'
    rows.append({'image': str(img_path), 'expected': expected, 'pred': pred, 'conf': conf, 'accuracy_label': label})

pred_df = pd.DataFrame(rows)
pred_csv = EVAL_OUT / 'sample_1000_real_field_predictions_full27_2.csv'
pred_df.to_csv(pred_csv, index=False)

summary = pred_df['accuracy_label'].value_counts().reindex(['true', 'close', 'false'], fill_value=0).to_frame().T
summary['total'] = len(pred_df)
summary['strict_acc'] = summary['true'] / summary['total']
summary['close_acc'] = (summary['true'] + summary['close']) / summary['total']

print('CSV 저장:', pred_csv)
display(summary[['true', 'close', 'false', 'total', 'strict_acc', 'close_acc']])

## 4. 기존 CCTV 이미지 폴더 추론

이미지를 노트북 코드 안에 base64로 넣으면 셀이 너무 길어지므로, 이제는 Drive 폴더에 있는 이미지를 읽어서 추론합니다.

기존 4장 이미지는 아래 폴더에 있다고 가정합니다.

`/content/drive/MyDrive/test_vehicle_images_four_cctv`

이미지가 없다면 다음 섹션의 업로드 셀로 다시 올리면 됩니다.


In [ ]:
FOUR_IMAGE_DIR = Path('/content/drive/MyDrive/test_vehicle_images_four_cctv')
FOUR_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED = {
    'burgundy_suv_side.png': 'korando_turismo',
    'white_suv_side.png': 'carnival_r',
    'black_sedan_side.png': 'grandeur_tg',
    'black_suv_front_side.png': 'rodius_rodius',
}

four_images = sorted([
    p for p in FOUR_IMAGE_DIR.glob('*')
    if p.is_file() and p.suffix.lower() in IMG_EXTS
])

print('이미지 폴더:', FOUR_IMAGE_DIR)
print('이미지 수:', len(four_images))
print('이미지:', [p.name for p in four_images])

In [ ]:
IMG_SIZE = globals().get('IMG_SIZE', 224)
CLS_WEIGHTS = BEST_WEIGHTS if BEST_WEIGHTS.exists() else LAST_WEIGHTS
model = YOLO(str(CLS_WEIGHTS), task='classify')

from PIL import Image as PILImage, ImageDraw, ImageFont
from IPython.display import display, Image as DisplayImage
import math


def save_prediction_contact_sheet(rows, image_dir, out_path, thumb_w=360, thumb_h=240, cols=2):
    if not rows:
        return None
    font = ImageFont.load_default()
    label_h = 78
    rows_n = math.ceil(len(rows) / cols)
    sheet = PILImage.new('RGB', (cols * thumb_w, rows_n * (thumb_h + label_h)), 'white')
    draw = ImageDraw.Draw(sheet)

    for idx, row in enumerate(rows):
        x = (idx % cols) * thumb_w
        y = (idx // cols) * (thumb_h + label_h)
        img_path = image_dir / row['image']
        try:
            img = PILImage.open(img_path).convert('RGB')
            img.thumbnail((thumb_w, thumb_h))
            px = x + (thumb_w - img.width) // 2
            py = y + (thumb_h - img.height) // 2
            sheet.paste(img, (px, py))
        except Exception as e:
            draw.text((x + 8, y + 8), f'Image load failed: {e}', fill='red', font=font)

        text_y = y + thumb_h + 6
        draw.text((x + 8, text_y), row['image'][:54], fill='black', font=font)
        draw.text((x + 8, text_y + 16), f"expected: {row['expected'] or '-'}", fill='black', font=font)
        draw.text((x + 8, text_y + 32), f"pred: {row['pred']} ({row['conf']:.4f})", fill='black', font=font)
        draw.text((x + 8, text_y + 48), f"label: {row['accuracy_label']}", fill='black', font=font)

    sheet.save(out_path)
    return out_path

rows = []
for img_path in four_images:
    expected = EXPECTED.get(img_path.name, '')
    result = model.predict(source=str(img_path), imgsz=IMG_SIZE, verbose=False)[0]

    top5_idx = [int(i) for i in result.probs.top5]
    top5 = [(result.names[i], float(result.probs.data[i])) for i in top5_idx]
    pred = top5[0][0]
    conf = top5[0][1]

    if not expected:
        label = 'unchecked'
    elif norm_name(pred) == norm_name(expected):
        label = 'true'
    elif is_close_match(pred, expected):
        label = 'close'
    else:
        label = 'false'

    rows.append({
        'image': img_path.name,
        'expected': expected,
        'pred': pred,
        'conf': conf,
        'accuracy_label': label,
        'top5': top5,
    })

four_df = pd.DataFrame(rows)
four_csv = FOUR_IMAGE_DIR / 'full27_2_four_cctv_predictions.csv'
four_df.to_csv(four_csv, index=False)
contact_sheet = FOUR_IMAGE_DIR / 'full27_2_four_cctv_contact_sheet.png'
save_prediction_contact_sheet(rows, FOUR_IMAGE_DIR, contact_sheet, cols=2)

print('CSV ??:', four_csv)
print('contact sheet ??:', contact_sheet)
display(four_df[['image', 'expected', 'pred', 'conf', 'accuracy_label', 'top5']])
if contact_sheet.exists():
    display(DisplayImage(filename=str(contact_sheet)))


## 5. 추가 CCTV 이미지 업로드 추론

채팅에 붙인 새 이미지처럼 파일이 노트북 안에 아직 없는 경우 이 섹션을 사용합니다.

실행하면 파일 선택창이 뜹니다. PNG/JPG 이미지를 여러 장 선택하면 Drive의 `test_vehicle_images_extra_cctv` 폴더에 저장하고 `full27_2`로 바로 추론합니다.

정답 라벨을 알고 있으면 `EXTRA_EXPECTED`에 파일명별로 적고, 모르면 비워둔 채 예측만 확인하면 됩니다.


In [ ]:
from google.colab import files

EXTRA_IMAGE_DIR = Path('/content/drive/MyDrive/test_vehicle_images_extra_cctv')
EXTRA_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
for name, data in uploaded.items():
    if Path(name).suffix.lower() in IMG_EXTS:
        (EXTRA_IMAGE_DIR / name).write_bytes(data)

print('저장 폴더:', EXTRA_IMAGE_DIR)
print('업로드 이미지:', sorted([p.name for p in EXTRA_IMAGE_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS]))

Saving 스크린샷 2026-07-23 135105.png to 스크린샷 2026-07-23 135105 (1).png
Saving 스크린샷 2026-07-23 135358.png to 스크린샷 2026-07-23 135358 (1).png
저장 폴더: /content/drive/MyDrive/test_vehicle_images_extra_cctv
업로드 이미지: ['User attachment.png', 'User attachment1.png', 'User attachment2.png', 'User attachment3.png', 'User attachment4.png', 'User attachment5.png', 'User attachment6.png', 'User attachment7.png', 'full27_2_extra_cctv_contact_sheet.jpg', '스크린샷 2026-07-23 135105 (1).png', '스크린샷 2026-07-23 135105.png', '스크린샷 2026-07-23 135358 (1).png', '스크린샷 2026-07-23 135358.png']


In [ ]:
IMG_SIZE = globals().get('IMG_SIZE', 224)
# ??? ?? ??? ?? ????. ??? ???? ??? ?? ?????.
# ?: EXTRA_EXPECTED = {'my_image.png': 'grandeur_tg'}
EXTRA_EXPECTED = {}

CLS_WEIGHTS = BEST_WEIGHTS if BEST_WEIGHTS.exists() else LAST_WEIGHTS
model = YOLO(str(CLS_WEIGHTS), task='classify')

from PIL import Image as PILImage, ImageDraw, ImageFont
from IPython.display import display, Image as DisplayImage
import math


def save_prediction_contact_sheet(rows, image_dir, out_path, thumb_w=360, thumb_h=240, cols=2):
    if not rows:
        return None
    font = ImageFont.load_default()
    label_h = 78
    rows_n = math.ceil(len(rows) / cols)
    sheet = PILImage.new('RGB', (cols * thumb_w, rows_n * (thumb_h + label_h)), 'white')
    draw = ImageDraw.Draw(sheet)

    for idx, row in enumerate(rows):
        x = (idx % cols) * thumb_w
        y = (idx // cols) * (thumb_h + label_h)
        img_path = image_dir / row['image']
        try:
            img = PILImage.open(img_path).convert('RGB')
            img.thumbnail((thumb_w, thumb_h))
            px = x + (thumb_w - img.width) // 2
            py = y + (thumb_h - img.height) // 2
            sheet.paste(img, (px, py))
        except Exception as e:
            draw.text((x + 8, y + 8), f'Image load failed: {e}', fill='red', font=font)

        text_y = y + thumb_h + 6
        draw.text((x + 8, text_y), row['image'][:54], fill='black', font=font)
        draw.text((x + 8, text_y + 16), f"expected: {row['expected'] or '-'}", fill='black', font=font)
        draw.text((x + 8, text_y + 32), f"pred: {row['pred']} ({row['conf']:.4f})", fill='black', font=font)
        draw.text((x + 8, text_y + 48), f"label: {row['accuracy_label']}", fill='black', font=font)

    sheet.save(out_path)
    return out_path

extra_rows = []
for img_path in sorted(EXTRA_IMAGE_DIR.glob('*')):
    if not img_path.is_file() or img_path.suffix.lower() not in IMG_EXTS:
        continue

    result = model.predict(source=str(img_path), imgsz=IMG_SIZE, verbose=False)[0]
    top5_idx = [int(i) for i in result.probs.top5]
    top5 = [(result.names[i], float(result.probs.data[i])) for i in top5_idx]
    pred = top5[0][0]
    conf = top5[0][1]
    expected = EXTRA_EXPECTED.get(img_path.name, '')

    if not expected:
        label = 'unchecked'
    elif norm_name(pred) == norm_name(expected):
        label = 'true'
    elif is_close_match(pred, expected):
        label = 'close'
    else:
        label = 'false'

    extra_rows.append({
        'image': img_path.name,
        'expected': expected,
        'pred': pred,
        'conf': conf,
        'accuracy_label': label,
        'top5': top5,
    })

extra_df = pd.DataFrame(extra_rows)
extra_csv = EXTRA_IMAGE_DIR / 'full27_2_extra_cctv_predictions.csv'
extra_df.to_csv(extra_csv, index=False)
contact_sheet = EXTRA_IMAGE_DIR / 'full27_2_extra_cctv_contact_sheet.png'
save_prediction_contact_sheet(extra_rows, EXTRA_IMAGE_DIR, contact_sheet, cols=2)

print('CSV ??:', extra_csv)
print('contact sheet ??:', contact_sheet)
display(extra_df[['image', 'expected', 'pred', 'conf', 'accuracy_label', 'top5']])
if contact_sheet.exists():
    display(DisplayImage(filename=str(contact_sheet)))
